# Job B (trained models) - STFPM / RD4AD / DRAEM / CSFlow on Deceuninck

**Model-major loop:** one model at a time over the full Deceuninck dataset. Start with `MODEL` in cell 4 set to `anomalib_stfpm`.

**Realistic time budget on T4 (Colab Pro):**
| Model | Full run | Sessions needed |
|---|---|---|
| `anomalib_stfpm` (100 ep) | ~2-3 h | 1 |
| `rd4ad` (200 ep)          | ~4-5 h | 1-2 |
| `anomalib_draem` (200 ep) | ~8-10 h | 2 |
| `anomalib_csflow` (240 ep)| ~15+ h | **does not fit Colab - use RunPod/HPC** |

**Recommended order on Colab:** `anomalib_stfpm` -> `rd4ad` -> `anomalib_draem`. Skip `anomalib_csflow` here.

**Outputs:** synced to `/content/drive/MyDrive/thesis_runs/jobB_trained/` per model. Resumable via `jobB_trained__<model>.done` markers.


In [ ]:
# 1. Mount Drive and verify the Deceuninck folders are visible.
# Deceuninck stores BMPs (line-camera native format), not JPEGs.
from google.colab import drive
drive.mount('/content/drive')

import os, glob
DATASET_DIR = '/content/drive/MyDrive/Deceuninck_dataset'
good_imgs    = glob.glob(os.path.join(DATASET_DIR, 'good', '*.bmp'))
defect_dirs  = sorted(d for d in glob.glob(os.path.join(DATASET_DIR, 'defects', '*')) if os.path.isdir(d))
defect_imgs  = sum(len(glob.glob(os.path.join(d, '*.bmp'))) for d in defect_dirs)
print(f'good/    : {len(good_imgs)} images')
print(f'defects/ : {len(defect_dirs)} subfolders, {defect_imgs} images total')
for d in defect_dirs:
    n = len(glob.glob(os.path.join(d, '*.bmp')))
    print(f'  - {os.path.basename(d):40s} {n} images')
assert len(good_imgs) > 0 and defect_imgs > 0, 'Dataset empty - check DATASET_DIR matches your Drive layout.'


In [ ]:
# 2. Clone (or pull) the thesis repo.
REPO_URL = 'https://github.com/LorenzSF/Real-time-visual-defect-detection.git'
REPO_DIR = '/content/Real-time-visual-defect-detection'
BRANCH   = 'main'

import os, subprocess
if os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', '-C', REPO_DIR, 'fetch', '--all'])
    subprocess.check_call(['git', '-C', REPO_DIR, 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])

print('Repo ready at', REPO_DIR)
print(subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--oneline']).decode().strip())


In [ ]:
# 3. Install Python dependencies. Same stack as the Job A trained notebook.
# anomalib >= 2.x (module path), transformers in 4.x (avoids 5.x breaking changes),
# kornia + einops are anomalib transitive deps Colab does not always pull.
# FrEIA is required by CSFlow only - installed defensively for completeness.
!pip install -q "anomalib>=2.2,<3" lightning "transformers>=4.46,<5" scikit-learn timm kornia einops FrEIA
!pip install -q -e /content/Real-time-visual-defect-detection --no-deps || echo '[note] editable install skipped (not required)'

import torch, anomalib, lightning, transformers
print('torch       ', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')
print('anomalib    ', anomalib.__version__)
print('lightning   ', lightning.__version__)
print('transformers', transformers.__version__)


In [ ]:
# 4. Pick the model for THIS session and launch.
#
# MODEL options (run one per session):
#   'anomalib_stfpm'   - fastest starting point on Colab
#   'rd4ad'            - moderate runtime, usually fits in one long session
#   'anomalib_draem'   - longer run, may need two sessions
#   'anomalib_csflow'  - too slow for Colab; use RunPod/HPC
#
# The driver stages the Deceuninck dataset to local SSD, then runs the
# selected model once and syncs the results back to Drive.

import os
os.environ['MODEL']       = 'anomalib_stfpm'
os.environ['DATASET_DIR'] = '/content/drive/MyDrive/Deceuninck_dataset'
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/thesis_runs/jobB_trained'
os.environ['WORK_DIR']    = '/content/work'
os.environ['REPO_DIR']    = '/content/Real-time-visual-defect-detection'
os.environ['CONFIG']      = '/content/Real-time-visual-defect-detection/src/benchmark_AD/configs/colab_trained_deceuninck.yaml'

# Ensure the Drive log dir exists before `tee` tries to append to it.
!mkdir -p {os.environ['RESULTS_DIR']}
!bash /content/Real-time-visual-defect-detection/scripts/run_jobB_trained_colab.sh 2>&1 | tee -a {os.environ['RESULTS_DIR']}/run_{os.environ['MODEL']}.log


## Resuming and switching models

- **Same model after disconnect:** rerun cells 1 -> 4. The `jobB_trained__<model>.done` marker is skipped automatically.
- **Switch to next model:** edit `MODEL` in cell 4 and rerun. Each model has its own marker, so the next run starts cleanly.
- **Force a rerun:** delete the marker on Drive, for example
  ```python
  !rm /content/drive/MyDrive/thesis_runs/jobB_trained/jobB_trained__anomalib_stfpm.done
  ```

## Troubleshooting

- **`Expected good/ and defects/ under ...`**: the Drive folder does not match the expected layout. Confirm `ls /content/drive/MyDrive/Deceuninck_dataset` shows `good/` and `defects/` at the top level.
- **`good/    : 0 images`**: the loader scans `*.bmp`. If your copy on Drive is JPEG, either re-export the dataset as BMP or relax the glob in cell 1 (the pipeline itself accepts `.bmp`, `.jpg`, `.jpeg`, `.png`, `.tif`, `.tiff`).
- **Drive copy is slow**: Deceuninck is many small BMPs and Drive throttles per-file I/O. If the copy stalls, consider zipping the folder once on Drive and adapting the driver to `unzip -q` on the local SSD like Job A does.
- **`CUDA out of memory` on DRAEM/CSFlow**: lower the corresponding `batch_size` in [colab_trained_deceuninck.yaml](../src/benchmark_AD/configs/colab_trained_deceuninck.yaml).
- **Session disconnected mid-run**: reopen the notebook, rerun cells 1-3, then rerun cell 4. Delete the marker first if you want to restart the pipeline itself.
- **Results look off**: check the per-model log under `RESULTS_DIR`. The pipeline prints per-model AUROC/F1 and any skipped-model warnings there.


In [ ]:
# 5. Release Colab resources once the run is finished and synced.
# This clears local caches first, then disconnects the runtime.
import gc

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

gc.collect()

from google.colab import runtime
runtime.unassign()
